# 16 — End-to-End Project Recipes: Classifier, Vision Transfer, Audio Classifier, Text Classifier, Mini Chatbot

Goal: provide complete, practical recipes you can adapt immediately.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## Projects in this notebook

1. Tabular classifier (MLP) with good training hygiene
2. Vision transfer learning (ResNet) skeleton
3. Audio classifier (mel + CNN) skeleton
4. Text classifier (bag-of-embeddings) in pure PyTorch
5. Mini chatbot (decoder-only LM) training + generation (educational scale)

Each recipe includes:
- model
- data pipeline
- training loop
- evaluation hooks
- checkpointing

## 1. Text classifier: embedding + pooling + MLP (pure PyTorch)

This is a strong baseline and teaches the key parts:
- vocab/tokenizer
- padding + masks
- embedding layers
- pooling strategies (mean/max/attention pooling)

In [ ]:

import torch, torch.nn as nn
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader

def tokenize(text):
    return re.findall(r"[a-z0-9]+|[^\s\w]", text.lower())

class Vocab:
    def __init__(self, texts, min_freq=1, specials=("<pad>","<unk>")):
        counts = Counter()
        for t in texts:
            counts.update(tokenize(t))
        self.itos = list(specials) + [w for w,c in counts.items() if c>=min_freq and w not in specials]
        self.stoi = {w:i for i,w in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
    def encode(self, text):
        return [self.stoi.get(w, self.unk_id) for w in tokenize(text)]

class TxtDS(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts; self.labels = labels; self.vocab = vocab
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return self.vocab.encode(self.texts[i]), int(self.labels[i])

def collate(batch, pad_id):
    seqs, labels = zip(*batch)
    lens = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    T = int(lens.max())
    x = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), T), dtype=torch.bool)
    for i,s in enumerate(seqs):
        x[i,:len(s)] = torch.tensor(s, dtype=torch.long)
        mask[i,:len(s)] = True
    return x.to(device), mask.to(device), torch.tensor(labels, dtype=torch.long).to(device)

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, d=128, num_classes=2, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d, padding_idx=pad_id)
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, num_classes))
    def forward(self, input_ids, mask):
        e = self.emb(input_ids)             # [B,T,d]
        mask_f = mask.unsqueeze(-1).float()
        pooled = (e * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        return self.mlp(pooled)

texts = ["I love PyTorch", "This is bad", "PyTorch is great", "I dislike bugs"]
labels = [1,0,1,0]
vocab = Vocab(texts)
ds = TxtDS(texts, labels, vocab)
dl = DataLoader(ds, batch_size=2, shuffle=True, collate_fn=lambda b: collate(b, vocab.pad_id))

model = TextClassifier(len(vocab.itos), pad_id=vocab.pad_id).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for epoch in range(50):
    model.train()
    for xb, mb, yb in dl:
        logits = model(xb, mb)
        loss = nn.CrossEntropyLoss()(logits, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

@torch.inference_mode()
def predict(text):
    ids = vocab.encode(text)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    m = torch.ones_like(x, dtype=torch.bool)
    p = torch.softmax(model(x,m), dim=-1)[0]
    return p.tolist()

predict("PyTorch is great")